In [1]:
!pip install -q datasets transformers accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

raw_datasets = load_dataset("ailsntua/QEvasion")

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def filter_long_sequences(examples):

    texts = [
        f"Question: {q} Answer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]

    tokenized = tokenizer(texts, truncation=False, padding=False)

    return [len(ids) <= 512 for ids in tokenized['input_ids']]

raw_datasets['train'] = raw_datasets['train'].filter(filter_long_sequences, batched=True)

train_df = raw_datasets['train'].to_pandas()

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=488,
    random_state=42,
    stratify=train_df['evasion_label']
)

raw_datasets['train'] = Dataset.from_pandas(train_df_split, preserve_index=False)
raw_datasets['validation'] = Dataset.from_pandas(val_df_split, preserve_index=False)

label2id = {
    "Explicit": 0,
    "Implicit": 1,
    "Dodging": 2,
    "General": 3,
    "Deflection": 4,
    "Partial/half-answer": 5,
    "Declining to answer": 6,
    "Claims ignorance": 7,
    "Clarification": 8
}

id2label = {v: k for k, v in label2id.items()}

print(raw_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Filter:   0%|          | 0/3448 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 1951
    })
    test: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 308
    })
    validation: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'anno

In [3]:
from transformers import DataCollatorWithPadding

def tokenize_function(examples):
    texts = [
        f"Question: {q} Answer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]

    tokenized = tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=512
    )

    tokenized['labels'] = [label2id[label] for label in examples['evasion_label']]

    return tokenized

tokenized_datasets = {}
tokenized_datasets['train'] = raw_datasets['train'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)
tokenized_datasets['validation'] = raw_datasets['validation'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['validation'].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1951 [00:00<?, ? examples/s]

Map:   0%|          | 0/488 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=9,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'f1': f1
    }

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_deberta",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    seed=42,
    fp16=True,
    report_to="none"
)

In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

)

/tmp/ipython-input-1360877873.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
trainer.train()

print("Fine tuning complete.")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.335100,3.838246,0.450820,0.393567
2,0.407500,4.189451,0.415984,0.396484
3,0.383100,4.360298,0.469262,0.377641
4,0.142600,4.598558,0.454918,0.404793
5,0.053800,4.568949,0.479508,0.423218


Fine tuning complete.


In [24]:

print("Subtask 2: Evasion Classification - Multi Ground Truth\n")

from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, f1_score

test_dataset_raw = load_dataset("ailsntua/QEvasion")['test']

def tokenize_test(examples):
    texts = [
        f"Question: {q} Answer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=512)

test_tokenized = test_dataset_raw.map(
    tokenize_test,
    batched=True,
    remove_columns=[col for col in test_dataset_raw.column_names
                    if col not in ['annotator1', 'annotator2', 'annotator3']]
)

predictions_logits = trainer.predict(test_tokenized)
predictions = np.argmax(predictions_logits.predictions, axis=-1)

predicted_labels = [id2label[pred] for pred in predictions]

correct = 0
total = len(test_dataset_raw)

for i in range(total):
    pred = predicted_labels[i]
    annotator_labels = [
        test_dataset_raw[i]['annotator1'],
        test_dataset_raw[i]['annotator2'],
        test_dataset_raw[i]['annotator3']
    ]

    if pred in annotator_labels:
        correct += 1

accuracy_multi_annotator = correct / total

all_labels = list(id2label.values())
label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}

n = len(test_dataset_raw)
y_true_binary = np.zeros((n, len(all_labels)), dtype=int)
y_pred_binary = np.zeros((n, len(all_labels)), dtype=int)

for i in range(n):
    ann_set = {
        test_dataset_raw[i]['annotator1'],
        test_dataset_raw[i]['annotator2'],
        test_dataset_raw[i]['annotator3']
    }

    for lbl in ann_set:
        if lbl in label_to_idx:
            y_true_binary[i, label_to_idx[lbl]] = 1

    pred = predicted_labels[i]
    if pred in label_to_idx:
        y_pred_binary[i, label_to_idx[pred]] = 1

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true_binary, y_pred_binary, average=None, labels=range(len(all_labels)), zero_division=0
)

macro_f1 = np.mean(f1s)
micro_f1 = f1_score(y_true_binary, y_pred_binary, average='micro', zero_division=0)

print(f"Accuracy: {accuracy_multi_annotator:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")

print("Per-class metrics:")
for i, lbl in enumerate(all_labels):
    print(f"   {lbl:<25s} P={precisions[i]:.2f} R={recalls[i]:.2f} F1={f1s[i]:.2f} Support={supports[i]:d}")

Subtask 2: Evasion Classification - Multi Ground Truth



Accuracy: 0.4805
Macro F1: 0.3594
Micro F1: 0.3558
Per-class metrics:
   Explicit                  P=0.62 R=0.53 F1=0.57 Support=115
   Implicit                  P=0.33 R=0.23 F1=0.27 Support=99
   Dodging                   P=0.54 R=0.32 F1=0.41 Support=96
   General                   P=0.43 R=0.13 F1=0.20 Support=113
   Deflection                P=0.27 R=0.14 F1=0.18 Support=51
   Partial/half-answer       P=0.12 R=0.07 F1=0.09 Support=14
   Declining to answer       P=0.43 R=0.18 F1=0.25 Support=17
   Claims ignorance          P=0.80 R=0.27 F1=0.40 Support=15
   Clarification             P=1.00 R=0.75 F1=0.86 Support=4
